# 检查 scTherapy 处理后数据

加载 Zenodo 处理后的 Seurat 对象或转换后的 AnnData，检查患者分布、细胞类型、恶性标签和表达矩阵。

In [ ]:
import sys
sys.path.insert(0, '..')
import os
from pathlib import Path
import pandas as pd
import numpy as np

# 查找处理后数据文件
processed_dir = Path('../data/processed')
rds_files = list(processed_dir.glob('*.rds'))
h5ad_files = list(processed_dir.glob('*.h5ad'))

print(f'RDS files: {[f.name for f in rds_files]}')
print(f'h5ad files: {[f.name for f in h5ad_files]}')

if not rds_files and not h5ad_files:
    print()
    print('No processed data found.')
    print('Run: python scripts/download_zenodo.py --list')
    print('Then: python scripts/download_zenodo.py --file <filename>')

In [ ]:
# 如果有 h5ad 文件，加载并检查
h5ad_files = list(Path('../data/processed').glob('*.h5ad'))
if h5ad_files:
    import anndata as ad
    adata = ad.read_h5ad(h5ad_files[0])
    print(f'Loaded: {h5ad_files[0].name}')
    print(f'Shape: {adata.shape}')
    print(f'obs columns: {list(adata.obs.columns)}')
    print()
    
    # 患者分布
    for col in ['patient', 'Patient', 'patient_id', 'sample']:
        if col in adata.obs.columns:
            print(f'Patient distribution ({col}):')
            print(adata.obs[col].value_counts())
            print()
            break
    
    # 细胞类型分布
    for col in ['cell_type', 'CellType', 'celltype', 'cluster']:
        if col in adata.obs.columns:
            print(f'Cell type distribution ({col}):')
            print(adata.obs[col].value_counts().head(20))
            print()
            break
    
    # 恶性标签
    for col in ['malignant', 'Malignant', 'is_malignant', 'cell_status']:
        if col in adata.obs.columns:
            print(f'Malignant labels ({col}):')
            print(adata.obs[col].value_counts())
            print()
            break

In [ ]:
# 如果只有 RDS 文件，提示使用 R 脚本
rds_files = list(Path('../data/processed').glob('*.rds'))
if rds_files and not h5ad_files:
    print('RDS files found. Use R script to inspect:')
    print(f'  Rscript scripts/inspect_seurat.R data/processed/{rds_files[0].name}')
    print()
    print('Or convert to h5ad using SeuratDisk in R:')
    print('  library(SeuratDisk)')
    print('  SaveH5Seurat(obj, filename = "output.h5Seurat")')
    print('  Convert("output.h5Seurat", dest = "h5ad")')

## 数据字典参考

关键元数据字段:
- patient: 患者编号
- cancer_type: 癌种 (AML/HGSC)
- cell_type: 细胞类型
- malignant: 是否恶性
- clone: 克隆编号
- nCount_RNA: 总 UMI 数
- nFeature_RNA: 检测基因数

详见 metadata/data_dictionary.md